In [2]:
import pandas as pd
docs = [
    "i love this product it works perfectly",
    "the service was terrible and very slow",
    "average experience nothing special",
    "excellent quality and fast delivery",
    "i am disappointed with the purchase",
    "not bad but could be better",
    "absolutely amazing highly recommend it",
    "worst experience ever",
    "it is okay for the price",
    "very satisfied with the customer support"
]

df_docs = pd.DataFrame({"text": docs})

generating embeddings

In [3]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embed_model.encode(df_docs["text"].tolist(),normalize_embeddings=True)

d:\Machine Learing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


retrival function using cosine similarity

In [4]:
import numpy as np
def retrieve(query,docs,doc_embeddings,top_k = 3):
    query_emb = embed_model.encode([query],normalize_embeddings=True)[0]
    
    scores = np.dot(doc_embeddings,query_emb)
    top_indices = scores.argsort()[::-1][:top_k]
    retrived_texts = [docs[i] for i in top_indices]
    return retrived_texts


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer,AutoModelForSeq2SeqLM

# t5 models are encoder-decoder architectures so use use AutoModelForSeq2SeqLM for loading it
llm_model_name = "google/flan-t5-small" 
tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(llm_model_name)

def generate_answer(query, retrieved_docs, max_new_tokens=100):
    context = "\n".join(retrieved_docs)
    prompt = f"Use the following context to answer the question:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return answer


In [8]:

query = "How was the delivery service?"
retrieved = retrieve(query, docs, doc_embeddings, top_k=3)

print("Retrieved Documents:")
for doc in retrieved:
    print("-", doc)

answer = generate_answer(query, retrieved)
print("\nLLM Answer:", answer)

Retrieved Documents:
- excellent quality and fast delivery
- the service was terrible and very slow
- very satisfied with the customer support

LLM Answer: (iii)
